In [1]:
import random
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

c:\Users\20878\26Winter\EEC289A-unsupervised-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


In [ ]:
import unicodedata
import re
from collections import Counter

def corpus_statistics(text):
    """
    Compute the unigram/single word frequency count (unigram),
    the bigram/word-tuple frequency count (bigram),
    the total number of words in the corpus (corpus_size), 
    and the number of unique-words in the corpus (vocab_size).
    """
    text = unicodedata.normalize("NFKC", text)
    tokens = re.findall(r"\b\w+\b", text.lower())
    corpus_size = len(tokens)
    token_tuples = [(tokens[i], tokens[i+1]) for i in range(corpus_size - 1)]
    unigram = Counter(tokens)
    bigram = Counter(token_tuples)
    vocab_size = len(unigram)
    return unigram, bigram, corpus_size, vocab_size 

def unigram_surprisal_score(tokenized_data, unigram, corpus_size, descending=False):
    """
    Compute the unigram surprisal score for each sequence in a batched tokenized data.
    Return:
        1. the score for each sequence in the original order;
        2. the re-ordered tokenized data based on each sequence's score in the ascending/descending order. 
    Unigram surprisal score:
        s = 1/T * Sigma_{t=1}^T (- log(p(x[t]))), x is a sequence, T is its length/word counts, 
        and p is the word frequency estimated from the corpus.
    """
    score_list = []
    p = {w: c / corpus_size for w, c in unigram.items()}
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    seqs = tokenizer.batch_decode(tokenized_data.to(dtype=torch.long))
    for seq in seqs:
        seq = unicodedata.normalize("NFKC", seq)
        tokens_inseq = re.findall(r"\b\w+\b", seq.lower())
        score = 0
        valid_tokens_inseq = 0
        for w in tokens_inseq:
            if w in unigram.keys():
                score += - np.log(p[w])
                valid_tokens_inseq += 1
            else:
                score += 0
        score /= valid_tokens_inseq
        score_list.append(score)
    perm = torch.argsort(torch.tensor(score_list), descending=descending)
    tokenized_data =  tokenized_data[perm]
    return score_list, tokenized_data

def bigram_surprisal_score(tokenized_data, bigram, unigram, vocab_size, 
                          smoothing=False, eps = 0.1, descending=False):
    """
    Compute the bigram surprisal score for each sequence in a batched tokenized data,
    and output:
        1. the score for each sequence in the original order;
        2. the re-ordered tokenized data based on each sequence's score in the ascending/descending order.
    Bigram surprisal score:
        s = 1/(T-1) * Sigma_{t=1}^{T-1} (- log(p(x_{t+1}|x_t))), x is a sequence, T is its length/word counts, 
        and p is the word-tuple frequency estimated from the corpus.
    Apply smoothing to the word frequency:
        p(x_{t+1}|x_t) = (bigram[(x_t, x_{t+1})] + eps) / (unigram[x_t] + eps * vocab_size)
    """
    score_list = []
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    seqs = tokenizer.batch_decode(tokenized_data.to(dtype=torch.long))
    for seq in seqs:
        seq = unicodedata.normalize("NFKC", seq)
        tokens_inseq = re.findall(r"\b\w+\b", seq.lower())
        score = 0
        valid_tokens_inseq = 0
        for i in range(len(tokens_inseq) - 1):
            h, w = tokens_inseq[i], tokens_inseq[i+1]
            if w not in unigram.keys() or (h, w) not in bigram.keys():
                score += 0
            else:
                if smoothing:
                    score += - np.log((bigram[(h, w)] + eps) / (unigram[w] + eps * vocab_size))
                else:
                    score += - np.log(bigram[(h, w)] / unigram[w])
                valid_tokens_inseq += 1
        score /= valid_tokens_inseq
        score_list.append(score)
    perm = torch.argsort(torch.tensor(score_list), descending=descending)
    tokenized_data = tokenized_data[perm]
    return score_list, tokenized_data


In [ ]:
def load_tokenized_wikitext(context_len=128, stride=64, pad_token=50303, train_subset=100000):
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split=["train", "test"])
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    corpus = "".join(ds[0]["text"][:train_subset])
    train_encodings = tokenizer((corpus) , return_tensors="pt")
    test_encodings = tokenizer("".join(ds[1]["text"]), return_tensors="pt")
    unigram, bigram, corpus_size, vocab_size = corpus_statistics(corpus)
    corpus_stats = dict(UNIGRAM=unigram, BIGRAM=bigram, N=corpus_size, V=vocab_size)

    tr_seq_len = train_encodings.input_ids.size(1)
    tr_B, tr_b = tr_seq_len // stride, 0
    x_train = torch.zeros((tr_B, context_len)).to(device)

    te_seq_len = test_encodings.input_ids.size(1)
    te_B, te_b = te_seq_len // stride, 0
    x_test = torch.zeros((te_B, context_len)).to(device)

    for begin_loc in tqdm(range(0, tr_seq_len, stride)):
        end_loc = min(begin_loc + context_len, tr_seq_len)
        if end_loc < tr_seq_len:
            x_train[tr_b] = train_encodings.input_ids[:, begin_loc:end_loc].to(device)
            tr_b += 1
        else:
            cur_row = train_encodings.input_ids[:, begin_loc:end_loc]
            new_row = torch.cat([cur_row,
                                 torch.full((1, context_len - cur_row.size(1)), pad_token)], dim=1)
            x_train[tr_b] = new_row
            tr_b += 1
            break
    
    x_train = x_train[:tr_b]

    for begin_loc in tqdm(range(0, te_seq_len, stride)):
        end_loc = min(begin_loc + context_len, te_seq_len)
        if end_loc < te_seq_len:
            x_test[te_b] = test_encodings.input_ids[:, begin_loc:end_loc].to(device)
            te_b += 1
        else:
            cur_row = test_encodings.input_ids[:, begin_loc:end_loc]
            new_row = torch.cat([cur_row,
                                 torch.full((1, context_len - cur_row.size(1)), pad_token)], dim=1)
            x_test[te_b] = new_row
            te_b += 1
            break

    x_test = x_test[:te_b]
    return x_train, x_test, corpus_stats

def make_causal_sequences(x: torch.Tensor, bos_token=50302):
    """
    x: (B, context_len) tokenized sequences
    Returns:
    x_in: (B, context_len) with bos_token=-2 added
    y: (B, context_len) targets
    """
    B, t = x.shape
    start = torch.full((B, 1), bos_token, dtype=torch.long)
    x_in = torch.cat([start, x[:, :-1]], dim=1).to(dtype=torch.long)
    y = x.to(dtype=torch.long)
    return x_in, y

def make_dataloaders(x_train, x_test, batch_size=128, bos_token=50302, train_subset=None, x_train_sorted=False):
    if train_subset is not None and train_subset < x_train.shape[0]:
        idx = torch.randperm(x_train.shape[0])[:train_subset]
        x_tr = x_train[idx]
    else:
        x_tr = x_train
    
    x_in_tr, y_tr = make_causal_sequences(x_tr, bos_token=bos_token)
    x_in_te, y_te = make_causal_sequences(x_test, bos_token=bos_token)

    train_ds = TensorDataset(x_in_tr, y_tr)
    test_ds = TensorDataset(x_in_te, y_te)

    num_workers = 2 if device.type == "cuda" else 0
    pin_memory = (device.type == "cuda")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=not x_train_sorted, num_workers=num_workers,
                              pin_memory=pin_memory, persistent_workers=(num_workers > 0))
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                             pin_memory=pin_memory, persistent_workers=(num_workers > 0))
    return train_loader, test_loader

In [3]:
from importlib import reload
import transformer_model
reload(transformer_model)

<module 'transformer_model' from 'c:\\Users\\20878\\26Winter\\EEC289A-unsupervised-learning\\notebooks\\transformer_model.py'>

In [ ]:
from transformer_model import *
import torch.nn.functional as F

def evaluate_nll_per_token(model: nn.Module, loader: DataLoader):
    model.eval()
    total = 0.0
    count = 0
    with torch.no_grad():
        for x_in, y in loader:
            x_in = x_in.to(device)
            y = y.to(device)
            _, loss = model(x_in, y)
            valid_token_nums = torch.sum(y > 0).item()
            total += float(loss.item() * valid_token_nums)
            count += valid_token_nums
    return total / count

def compute_singular_entropy(model: nn.Module):
    _, S, _ = torch.linalg.svd(model.lm_head, full_matrices=False)
    Lambda = torch.diag(S @ S)
    p = Lambda / torch.sum(Lambda)
    return (p.T @ torch.log(p) + torch.log(p.shape[0])).item()

def compute_anisotropy(model: nn.Module, idx: torch.tensor):
    H = model(idx, return_features=True)
    det_H = torch.linalg.det(H)
    H = F.normalize(H, dim=1)
    Sim = H @ H.T
    mask = torch.eye(H.shape[0], dtype=torch.bool)
    Sim = Sim.masked_fill(mask, 0)
    return (torch.sum(Sim) / (det_H**2 - det_H)).item()

def train_LM(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, model_tag: str,
             epochs=3, lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1, grad_norm_clip=1.0):
    opt = model.configure_optimizers(lr=lr, betas=betas, weight_decay=weight_decay)
    history = {"train_loss": [], "test_nll": []}
    steps = 0

    epochs = epochs
    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        n_batches = 0
        pbar = tqdm(train_loader, desc=f"epoch {ep}/{epochs}")
        for x_in, y in pbar:
            x_in, y = x_in.to(device), y.to(device)

            opt.zero_grad(set_to_none=True)
            _, loss = model(x_in, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm_clip)
            opt.step()

            running += float(loss.item())
            n_batches += 1
            pbar.set_postfix(loss=running / n_batches)
            if steps // 100 == 0:
                sing_en = compute_singular_entropy(model)
                aniso = compute_anisotropy(model, x_in)
                torch.save({
                    'step': steps,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'singular_entropy': sing_en,
                    'anisotropy': aniso,
                }, f"ckpt_{model_tag}_{steps}.pth")
            steps += 1
        
        train_loss = running / max(1, n_batches)
        test_nll = evaluate_nll_per_token(model, test_loader)
        history["train_loss"].append(train_loss)
        history["test_nll"].append(test_nll)
        print(f"epoch {ep}: train_loss={train_loss:.4f} | test_null/token={test_nll:.4f}")
    
    return history
            



In [ ]:
import matplotlib.pyplot as plt
PAD_TOKEN = 50303
BOS_TOKEN = 50302
CONTEXT_LEN = 128
STRIDE = 64
BATCH_SIZE = 64
TRAIN_SUBSET = 1000 # reference: 100000
CURRICULUM = 0 # 0: random, -1: anti-curriculum, 1: curriculum
SCORE = "unigram" # "unigram" or "bigram"

MODEL_KWARGS = dict(
    vocab_size=50304,
    block_size=CONTEXT_LEN,
    n_layer=2,
    n_embd=256,
    n_head=4,
    bias=False,
    dropout=0.1,
    pad_token=PAD_TOKEN
)
TRAINER_KWARGS = dict(
    epochs=3, 
    lr=3e-4, 
    betas=(0.9, 0.95), 
    weight_decay=0.1, 
    grad_norm_clip=1.0
)

x_train, x_test, corpus_stats = load_tokenized_wikitext(context_len=CONTEXT_LEN, stride=STRIDE, train_subset=TRAIN_SUBSET, pad_token=PAD_TOKEN)
if CURRICULUM != 0:
    train_sorted = True
    if CURRICULUM == -1:
        descending = True
    else:
        descending = False
    if SCORE == "unigram":
        score_list, x_train = unigram_surprisal_score(x_train, corpus_stats["unigram"],
                                                  corpus_stats["N"], descending=descending)
    elif SCORE == "bigram":
        score_list, x_train = bigram_surprisal_score(x_train, corpus_stats["bigram"], corpus_stats["unigram"],
                                                  corpus_stats["V"], descending=descending)
else:
    train_sorted = False
train_loader, test_loader = make_dataloaders(x_train, x_test, batch_size=BATCH_SIZE, bos_token=BOS_TOKEN, x_train_sorted=train_sorted)

Token indices sequence length is longer than the specified maximum sequence length for this model (61816 > 1024). Running this sequence through the model will result in indexing errors
100%|█████████▉| 4425/4427 [00:00<00:00, 45299.28it/s]


In [6]:
model = GPT(**MODEL_KWARGS).to(device)
history = train_LM(model, train_loader, test_loader, **TRAINER_KWARGS)

plt.figure()
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], marker="o")
plt.xlabel("epoch")
plt.ylabel("train CE (mean)")
plt.title("GPT training curve")
plt.grid(True)
plt.show()

number of parameters: 14.48M


epoch 1/3:  31%|███▏      | 5/16 [00:50<01:51, 10.18s/it, loss=10.5]


KeyboardInterrupt: 

In [33]:
# embedding = nn.Embedding(10, 3, padding_idx=1)
# input = torch.LongTensor([[1, 2, 4, 0], [4, 3, 2, 10]])
# embedding(input)
x_train = x_train.to(dtype=torch.long)
x_train[x_train == 796]

tensor([796, 796, 796,  ..., 796, 796, 796])